# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row represents one content item for one client, identified by the combination of `client_hash_id` and `content_hash_id`.

The raw daily performance table contains daily observations for each client-content pair. For the clustering analysis, we will aggregate these daily observations into page-level features.

### Time window

For the first clustering analysis, I will use a defined historical window rather than mixing observations from different periods. The feature window will be used to calculate the performance characteristics of each client-content pair.

The final month will be treated as a sealed outcome/test period and will not be used to develop the feature logic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature fields

The initial clustering features will be limited to measurable search-performance characteristics:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- search-demand related metrics available in the warehouse
- derived CTR where appropriate

These features describe how a content item performs and allow the clustering algorithm to identify different performance patterns.

### Label

There is no supervised target label for this task. This is an unsupervised clustering problem. The model will create a `cluster_id` for each client-content pair.

### Context fields

`client_hash_id` and `content_hash_id` identify the unit being analyzed. They are identifiers and will be retained for interpretation and joining results, but they should not be treated as numerical clustering features.

### Excluded fields

Future performance measurements are deliberately excluded from the feature set. Any clicks, impressions, CTR, position or other information from after the feature/decision window would leak future information into the analysis.

Client-identifying information, URLs, queries, keywords and other private/raw identifying information are also excluded from outputs.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



In [1]:
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")

    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

HF_TOKEN = get_hf_token()

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("Connected to DuckDB.")
print("Using March 2026 as the development window.")

Connected to DuckDB.
Using March 2026 as the development window.


In [2]:
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_client_content_pairs,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
        AS unique_client_content_days
FROM {FACT_DAILY}
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_pairs,unique_client_content_days
0,9841378,331437,9841378


In [3]:
date_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FACT_DAILY}
""").df()

display(date_check)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [4]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS unavailable_or_null_rows
FROM {FACT_DAILY}
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,unavailable_or_null_rows
0,9841378,3611061,6230317


### Data limits

This analysis has several important limitations:

1. **History is unbalanced across content and clients.**
   Not every client-content pair has the same amount of historical data, so comparisons should not assume identical observation history.

2. **GSC availability is uneven.**
   A large proportion of daily warehouse rows do not have GSC data available. Therefore, GSC-derived features such as impressions, clicks, CTR, average position, and position volatility should only be calculated from rows where the relevant GSC data is available.

3. **March 2026 is a development window, not a final test set.**
   We use March 2026 as the mid-panel development month. The final month of the warehouse should remain sealed for final evaluation.

4. **This clustering analysis is descriptive, not causal.**
   A cluster represents a measurable performance archetype. It does not prove that belonging to that archetype causes better or worse performance.

5. **The analysis uses a fixed monthly window.**
   Since the initial clustering features are calculated from March 2026, this notebook does not claim that the resulting archetypes remain unchanged across other months.

6. **Identifiers are not model features.**
   Client and content IDs are used to identify and group observations, but they are deliberately excluded from clustering because they are identifiers rather than meaningful performance signals.

In [5]:
client_availability = con.sql(f"""
    SELECT
        COUNT(DISTINCT client_hash_id) AS total_clients,

        COUNT(DISTINCT CASE
            WHEN gsc_data_available IS TRUE
            THEN client_hash_id
        END) AS clients_with_gsc_data,

        COUNT(DISTINCT CASE
            WHEN gsc_data_available IS NOT TRUE
            THEN client_hash_id
        END) AS clients_without_gsc_data

    FROM {FACT_DAILY}
""").df()

display(client_availability)

,total_clients,clients_with_gsc_data,clients_without_gsc_data
0,55,47,55


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.